# Author profiles
The form of each named poet, worked out from their training poems:
1. line length: the 25th-75th percentile of syllables per line
2. rhyme scheme: AABB, ABAB, ABCB or none
3. λ: 1 for poets who don't rhyme (no rhyme pressure), 0.5 for now for the rest (the λ sweep sets it later)
4. showcase poets: the poet with the most lines in each scheme group

Uses the rhyme tools from `rhyme.ipynb`. Output: `data/author_profiles.csv`

In [1]:
%%capture
%run rhyme.ipynb

In [2]:
from functools import lru_cache
import numpy as np

named = pd.read_csv(DATA + "train_test/named_poets.csv")
# by author, not header: 15% of the pieces have their header switched to "Unknown" but it's still their poem
train = pd.read_parquet(DATA + "train_test/train.parquet", columns=["author", "text"])
train = train[train["author"].isin(named["author"])]

# the words of every line, one list per training piece
pieces = []
for poet, text in zip(train["author"], train["text"]):
    lines = [line for line in poem_lines(text) if not re.fullmatch(r"[ivxlc]+\.?", line.strip().lower())]  # skip "II."
    lines = [words_of(line) for line in lines]
    pieces.append((poet, [w for w in lines if w]))

print(len(pieces), "pieces from", train["author"].nunique(), "poets")
missing = named[~named["author"].isin(train["author"])]
print(len(missing), "named poets have no training pieces:")
print(missing.to_string(index=False))

82033 pieces from 471 poets
15 named poets have no training pieces:
                author  distinct_lines
  Edward Bulwer Lytton           20109
        Thomas Cowherd           11009
         Rebekah Smith            3299
  Eugenia Dunlap Potts            2813
        Mathilde Blind            2467
      H. Lavinia Baily            1603
George Spencer Cautley            1557
     John Graham Bower            1135
         William Young            1078
            Evan Lloyd             994
     Margaret Widdemer             867
    Kate Simpson Hayes             807
      Wilbur D. Nesbit             777
           Thomas Gray             745
       Furnley Maurice             594


Those poets only have one or two long books and they all went into val/test in the 90/5/5 split,
so the model never sees their name in training. We can't generate for them, they get no profile.

## 1. Line length

In [3]:
syl = defaultdict(list)
for poet, lines in pieces:
    syl[poet] += [sum(syllables(w) for w in words) for words in lines]

profiles = pd.DataFrame({"poet": list(syl)})
profiles["lines"] = [len(syl[p]) for p in profiles["poet"]]
profiles["syl_min"] = [int(np.percentile(syl[p], 25)) for p in profiles["poet"]]
profiles["syl_max"] = [int(np.percentile(syl[p], 75)) for p in profiles["poet"]]
profiles = profiles.set_index("poet")

# share of line ends we have no pronunciation for (old spellings), their rhymes can't be found
unknown = defaultdict(list)
for poet, lines in pieces:
    unknown[poet] += [not phones(words[-1]) for words in lines]
profiles["unknown_ends"] = [round(np.mean(unknown[p]), 2) for p in profiles.index]

print(profiles.loc[["Alexander Pope", "Emily Dickinson", "John Milton", "Walt Whitman"]])

                 lines  syl_min  syl_max  unknown_ends
poet                                                  
Alexander Pope   23971       10       10          0.04
Emily Dickinson   5084        6        8          0.02
John Milton      17535       10       10          0.11
Walt Whitman      9855       11       19          0.03


Pope and Milton write 10-syllable lines, Dickinson alternates 8 and 6, Whitman's lines are long and all over the place.

## 2. Rhyme scheme
Cut the line ends into blocks of 4 and label each block:
- AABB: lines 1-2 and 3-4 rhyme
- ABAB: lines 1-3 and 2-4 rhyme
- ABCB: only lines 2-4 rhyme
- none: anything else

"Rhyme" here means a score of at least 0.9 (perfect, or sea / melody). The data has no blank lines between
stanzas, so we don't know where a stanza starts. We try all 4 starting points in each piece and keep the one
that gives the most rhymed blocks.

In [4]:
@lru_cache(maxsize=None)
def rhymes(a, b):
    return rhyme(a, b) >= 0.9


def label(block):
    a, b, c, d = block
    if rhymes(a, b) and rhymes(c, d):
        return "AABB"
    if rhymes(a, c) and rhymes(b, d):
        return "ABAB"
    if rhymes(b, d):
        return "ABCB"
    return "none"


counts = defaultdict(Counter)
for poet, lines in pieces:
    ends = [words[-1] for words in lines]
    best = None
    for start in range(4):
        labels = [label(ends[i:i + 4]) for i in range(start, len(ends) - 3, 4)]
        if best is None or sum(x != "none" for x in labels) > sum(x != "none" for x in best):
            best = labels
    counts[poet].update(best)

blocks = pd.DataFrame(counts).T.fillna(0).astype(int)[["AABB", "ABAB", "ABCB", "none"]]
share = blocks.div(blocks.sum(axis=1), axis=0)
print(share.loc[["Alexander Pope", "Emily Dickinson", "John Milton", "Walt Whitman"]].round(2))

                 AABB  ABAB  ABCB  none
Alexander Pope   0.55  0.02  0.03  0.41
Emily Dickinson  0.01  0.00  0.47  0.51
John Milton      0.05  0.02  0.03  0.90
Walt Whitman     0.00  0.00  0.03  0.97


How much do poets rhyme at all? (share of blocks that got any rhymed label)
Poets we know don't rhyme (Milton, Whitman) are at 0.03-0.10, so that's what chance looks like.

In [5]:
rhymed = 1 - share["none"]
print(rhymed.describe().round(2))

known = ["Alexander Pope", "Emily Dickinson", "John Milton", "Walt Whitman", "John Keats", "Robert Burns",
         "William Shakespeare", "Edgar Allan Poe", "Alfred Lord Tennyson", "Robert Browning", "William Blake",
         "Percy Bysshe Shelley", "George Gordon Byron", "Rudyard Kipling", "Thomas Hardy"]
print(share.loc[known].assign(rhymed=rhymed).round(2))

count    471.00
mean       0.49
std        0.21
min        0.01
25%        0.37
50%        0.52
75%        0.64
max        1.00
Name: none, dtype: float64
                      AABB  ABAB  ABCB  none  rhymed
Alexander Pope        0.55  0.02  0.03  0.41    0.59
Emily Dickinson       0.01  0.00  0.47  0.51    0.49
John Milton           0.05  0.02  0.03  0.90    0.10
Walt Whitman          0.00  0.00  0.03  0.97    0.03
John Keats            0.34  0.05  0.08  0.52    0.48
Robert Burns          0.07  0.05  0.28  0.61    0.39
William Shakespeare   0.06  0.27  0.19  0.48    0.52
Edgar Allan Poe       0.14  0.09  0.15  0.61    0.39
Alfred Lord Tennyson  0.06  0.05  0.10  0.78    0.22
Robert Browning       0.11  0.03  0.07  0.79    0.21
William Blake         0.25  0.04  0.12  0.60    0.40
Percy Bysshe Shelley  0.08  0.07  0.14  0.72    0.28
George Gordon Byron   0.16  0.20  0.17  0.47    0.53
Rudyard Kipling       0.17  0.16  0.27  0.40    0.60
Thomas Hardy          0.08  0.19  0.26  0.47    0.

Rule: a poet rhymes if at least 30% of their blocks rhyme, and then their scheme is the rhymed label they
use most. Below that it's "none".

Not perfect: Tennyson and Shelley do rhyme, but in forms we don't check for (In Memoriam is ABBA) and mixed with a
lot of blank verse, so they end up "none". Fine for now, it only means less rhyme pressure for them.

In [6]:
MIN_RHYMED = 0.3

profiles["scheme"] = share[["AABB", "ABAB", "ABCB"]].idxmax(axis=1).where(rhymed >= MIN_RHYMED, "none")
profiles["rhymed"] = rhymed.round(2)
profiles["lam"] = np.where(profiles["scheme"] == "none", 1.0, 0.5)

print(profiles["scheme"].value_counts())
print(profiles.loc[known, ["scheme", "rhymed"]])

scheme
ABCB    219
AABB    138
none     84
ABAB     30
Name: count, dtype: int64
                     scheme  rhymed
poet                               
Alexander Pope         AABB    0.59
Emily Dickinson        ABCB    0.49
John Milton            none    0.10
Walt Whitman           none    0.03
John Keats             AABB    0.48
Robert Burns           ABCB    0.39
William Shakespeare    ABAB    0.52
Edgar Allan Poe        ABCB    0.39
Alfred Lord Tennyson   none    0.22
Robert Browning        none    0.21
William Blake          AABB    0.40
Percy Bysshe Shelley   none    0.28
George Gordon Byron    ABAB    0.53
Rudyard Kipling        ABCB    0.60
Thomas Hardy           ABCB    0.53


## 3. Showcase poets
The poet with the most training lines in each scheme group (the top 3 shown, the first one is picked).
Poets with more than 10% unknown line ends are left out (mostly Middle English like John Gower), their
scheme can't be trusted: Gower rhymes, but we can't see it.

In [7]:
print(profiles.sort_values("unknown_ends", ascending=False).head(5)[["lines", "scheme", "rhymed", "unknown_ends"]])

by_size = profiles[profiles["unknown_ends"] <= 0.1].sort_values("lines", ascending=False)
print(by_size.groupby("scheme").head(3).sort_values("scheme"))

showcase = by_size.groupby("scheme").head(1)
print(showcase.index.tolist())

                          lines scheme  rhymed  unknown_ends
poet                                                        
John Lydgate               1406   none    0.09          0.64
John Gower                27283   none    0.04          0.59
Geoffrey Chaucer           8929   none    0.14          0.48
Chauncey Brewster Tinker    824   none    0.07          0.46
R. W. Chambers              723   none    0.05          0.38
                            lines  syl_min  syl_max  unknown_ends scheme  \
poet                                                                       
Algernon Charles Swinburne  42005        8       11          0.04   AABB   
Thomas Moore                39773        8       10          0.04   AABB   
Ella Wheeler Wilcox         34802        8       11          0.01   AABB   
George Gordon Byron         33216       10       11          0.04   ABAB   
Henry Kendall               10787        8       11          0.02   ABAB   
Robert William Service      10751        

In [8]:
profiles.sort_values("lines", ascending=False).to_csv(DATA + "author_profiles.csv")
print("saved", len(profiles), "profiles")

saved 471 profiles
